# Project 02 Analysis — Neural Field Theory

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/praveen-dedigamage/Cuda-For-Computational-Neuroscience/blob/main/projects/project_02_neural_field_theory/analysis.ipynb)

In [ ]:
!nvidia-smi
!nvcc -O2 -o neural_field neural_field.cu -lcufft -lm
!./neural_field 256 500

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load snapshots
snapshots = {}
with open('neural_field_snapshots.txt') as f:
    current_t, rows = None, []
    for line in f:
        if line.startswith('#'):
            if current_t is not None:
                snapshots[current_t] = np.array(rows)
            current_t = float(line.split('=')[1].split()[0])
            rows = []
        else:
            rows.append(list(map(float, line.split())))
    if current_t is not None:
        snapshots[current_t] = np.array(rows)

times = sorted(snapshots.keys())
print(f"Snapshots: {times} ms, Grid: {snapshots[times[0]].shape}")

# 2D activation maps
fig, axes = plt.subplots(1, len(times), figsize=(4*len(times), 4))
if len(times) == 1: axes = [axes]

for ax, t in zip(axes, times):
    U = snapshots[t]
    im = ax.imshow(U, cmap='RdBu_r', vmin=0, vmax=1, aspect='auto')
    ax.set_title(f't = {t:.0f} ms', fontsize=12)
    ax.set_xlabel('x (mm)'); ax.set_ylabel('y (mm)')
    plt.colorbar(im, ax=ax, label='u')

plt.suptitle('Wilson-Cowan Neural Field: Excitatory Activity', fontsize=14)
plt.tight_layout()
plt.savefig('p02_field_maps.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Dispersion relation: theoretical analysis
import numpy as np
import matplotlib.pyplot as plt

# Parameters (must match neural_field.cu)
tau_E = 10.0  # ms
A_EE, sigma_EE = 4.0, 2.0
B_EE, sigma_IE = 2.0, 8.0

# Fourier transform of Mexican-hat kernel: Gaussian FT is also Gaussian
# FT(A * Gauss(sigma)) = A * 2π * sigma² * Gauss(1/sigma)
# For 2D: FT(A * exp(-r²/2σ²)) = A * 2π * σ² * exp(-2π²σ²k²)

k = np.linspace(0, 1.0, 200)  # spatial frequency (cycles/mm)
K_hat = (A_EE * 2*np.pi * sigma_EE**2 * np.exp(-2*np.pi**2 * sigma_EE**2 * k**2)
       - B_EE * 2*np.pi * sigma_IE**2 * np.exp(-2*np.pi**2 * sigma_IE**2 * k**2))

# Sigmoid derivative at fixed point u* ≈ 0.5: S'(0) = 0.25
S_prime = 0.25

# Growth rate: sigma(k) = -1/tau_E + K_hat(k) * S'
sigma_k = -1.0/tau_E + K_hat * S_prime

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Mexican-hat kernel in k-space
axes[0].plot(k, K_hat, 'b-', lw=2)
axes[0].axhline(0, color='k', lw=0.8)
axes[0].set_xlabel('Spatial frequency k (cycles/mm)', fontsize=12)
axes[0].set_ylabel('Kernel FT K̂(k)', fontsize=12)
axes[0].set_title('Mexican-Hat Kernel in Frequency Space', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Dispersion relation
axes[1].plot(k, sigma_k, 'r-', lw=2)
axes[1].axhline(0, color='k', lw=1.5, linestyle='--')
axes[1].fill_between(k, sigma_k, 0, where=sigma_k>0,
                     color='red', alpha=0.2, label='Unstable (grows)')
axes[1].fill_between(k, sigma_k, 0, where=sigma_k<0,
                     color='blue', alpha=0.2, label='Stable (decays)')

if np.any(sigma_k > 0):
    k_max = k[np.argmax(sigma_k)]
    lambda_max = 1.0/k_max if k_max > 0 else np.inf
    axes[1].axvline(k_max, color='orange', linestyle=':', lw=2,
                    label=f'k_max = {k_max:.3f} → λ = {lambda_max:.1f} mm')
    print(f"Most unstable wavelength: {lambda_max:.1f} mm")
else:
    print("System is linearly stable — patterns are noise-driven")

axes[1].set_xlabel('Spatial frequency k (cycles/mm)', fontsize=12)
axes[1].set_ylabel('Growth rate σ(k) (ms⁻¹)', fontsize=12)
axes[1].set_title('Dispersion Relation: Growth Rate vs Spatial Frequency', fontsize=12)
axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('p02_dispersion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Spatial frequency content of final snapshot
import numpy as np
import matplotlib.pyplot as plt

U_final = snapshots[max(times)]
N = U_final.shape[0]

# 2D FFT and radially averaged power spectrum
U_fft = np.fft.fft2(U_final - U_final.mean())
P = np.abs(np.fft.fftshift(U_fft))**2 / N**2

# Radial average
cy, cx = N//2, N//2
y, x = np.ogrid[-cy:N-cy, -cx:N-cx]
r = np.sqrt(x**2 + y**2).astype(int)

max_r = N//2
radial_power = np.array([P[r == ri].mean() for ri in range(max_r)])
k_vals = np.arange(max_r) / N  # cycles per pixel

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im = axes[0].imshow(np.log10(P + 1e-12), cmap='hot', aspect='auto')
plt.colorbar(im, ax=axes[0], label='log₁₀ Power')
axes[0].set_title('2D Power Spectrum of Final Activity Pattern', fontsize=11)

axes[1].semilogy(k_vals, radial_power, 'b-', lw=2)
if radial_power.max() > 0:
    k_peak = k_vals[np.argmax(radial_power[1:]) + 1]
    axes[1].axvline(k_peak, color='r', linestyle='--',
                    label=f'Peak: k={k_peak:.3f} cycles/px → λ={1/k_peak:.0f} px')
axes[1].set_xlabel('Spatial frequency (cycles/pixel)', fontsize=12)
axes[1].set_ylabel('Radial power (log scale)', fontsize=12)
axes[1].set_title('Radially Averaged Power Spectrum', fontsize=11)
axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('p02_spatial_spectrum.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Performance scaling
import subprocess
import numpy as np
import matplotlib.pyplot as plt

grids = [64, 128, 256]
throughputs = []
realtime_factors = []

for G in grids:
    result = subprocess.run(['./neural_field', str(G), '200'],
                            capture_output=True, text=True)
    for line in result.stdout.split('\n'):
        if 'Throughput' in line:
            throughputs.append(float(line.split()[1]))
        if 'real-time' in line:
            realtime_factors.append(float(line.split()[-1].replace('x','')))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
n_pixels = [g*g for g in grids]

axes[0].loglog(n_pixels, throughputs, 'b-o', lw=2, markersize=8)
axes[0].set_xlabel('Grid size (pixels)', fontsize=12)
axes[0].set_ylabel('M pixel-steps/s', fontsize=12)
axes[0].set_title('Throughput vs Grid Size', fontsize=12)
axes[0].set_xticks(n_pixels)
axes[0].set_xticklabels([f'{g}²' for g in grids])
axes[0].grid(True, which='both', alpha=0.3)

axes[1].bar([f'{g}²' for g in grids], realtime_factors, color='coral', alpha=0.8)
axes[1].axhline(1, color='k', linestyle='--', label='Real-time')
axes[1].set_xlabel('Grid size', fontsize=12)
axes[1].set_ylabel('Real-time speedup', fontsize=12)
axes[1].set_title('Simulation vs Real Time', fontsize=12)
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

for G, tp, rt in zip(grids, throughputs, realtime_factors):
    print(f"{G}×{G}: {tp:.0f} M pixel-steps/s, {rt:.0f}× real-time")